In [1]:
import numpy as np
import pandas as pd

In [2]:
# Load in mgref file 
# MGREF is the spatial cross-reference files for nonpoint (stationary area)/nonroad and onroad mobile sources.
# https://www.cmascenter.org/smoke/documentation/4.5/manual_smokev45.pdf
file= '../AADT_comb_surrogate/mgref_onroad_MOVES3_18nov2022_v4.txt'
colheaders= ['FIPS','SCC','Surrogate']
# Including dtype = str here to retain the first columns set of zeros 
df = pd.read_csv(file, sep=';', header=None,  comment="#", dtype=str)
# Adding column headers to help make a bit more sense. 
df.columns= ['FIPS','SCC','Surrogate']

In [3]:
# SCCs in this file follow the following format:
# First three digits are always 220
# The fourth digit is the Fuel type (01- gas, 02-diesel, 03-CNG, 04- Petroleum gasm 05- Ethonal, 09- EV )
# Fifth and six digits are the vehicle codes. HDVs are 41,42,43,51,52,53,61,62 - Refer to MOVES TSD for specific vehicle types
# 7th and 8th digits are the Road types (01-off-road)
# 9th and 10th are process codes 
# Processes included in file are: 
# 40 - Breakware/tireware, 
# 53 - All extended idling exhaust, 
# 62 - All Refueling
# 72 - All exhaust and evaporative except refueling and hoteling
# 91 - Auxiliary Power Exhaust
# 92 - ONI 

# Here we want to break the SCC into components to more easily identify the vehicle and road types for HDV ONI.
# First, convert the number to string and index every 2 digits for mobile source, fuel type, etc. 
df['SCC1'] = df['SCC'].astype(str)
df['SCC_mobileSource'] = df['SCC1'].str[0:2]
df['SCC_fuelType'] = df['SCC1'].str[2:4]
df['SCC_VehicleCode'] = df['SCC1'].str[4:6]
df['SCC_RoadType'] = df['SCC1'].str[6:8]
df['SCC_ProcessCode'] = df['SCC1'].str[8:10]

In [4]:
df[(df.SCC_VehicleCode.isin(['52','53'])) & (df['SCC_RoadType'] == '01') & (df['SCC_ProcessCode'] == '92')]

,FIPS,SCC,Surrogate,SCC1,SCC_mobileSource,SCC_fuelType,SCC_VehicleCode,SCC_RoadType,SCC_ProcessCode
387,000000,2201520192,306,2201520192,22,01,52,01,92
388,000000,2201530192,306,2201530192,22,01,53,01,92
394,000000,2202520192,306,2202520192,22,02,52,01,92
395,000000,2202530192,306,2202530192,22,02,53,01,92
484,000000,2203520192,306,2203520192,22,03,52,01,92
499,000000,2203530192,306,2203530192,22,03,53,01,92


In [5]:
df[(df.SCC_VehicleCode.isin(['61','62'])) & (df['SCC_RoadType'] == '01') & (df['SCC_ProcessCode'] == '92')]

,FIPS,SCC,Surrogate,SCC1,SCC_mobileSource,SCC_fuelType,SCC_VehicleCode,SCC_RoadType,SCC_ProcessCode
390,000000,2201610192,306,2201610192,22,01,61,01,92
397,000000,2202610192,306,2202610192,22,02,61,01,92
398,000000,2202620192,306,2202620192,22,02,62,01,92
529,000000,2203610192,306,2203610192,22,03,61,01,92
566,000000,2203620192,306,2203620192,22,03,62,01,92


In [6]:
df[(df.SCC_VehicleCode.isin(['62'])) & (df['SCC_RoadType'] == '01') & (df['SCC_ProcessCode'] == '53')]

,FIPS,SCC,Surrogate,SCC1,SCC_mobileSource,SCC_fuelType,SCC_VehicleCode,SCC_RoadType,SCC_ProcessCode
0,000000,2202620153,205,2202620153,22,02,62,01,53
371,008000,2202620153,242,2202620153,22,02,62,01,53
373,023000,2202620153,242,2202620153,22,02,62,01,53
375,034000,2202620153,242,2202620153,22,02,62,01,53
377,036000,2202620153,242,2202620153,22,02,62,01,53
542,000000,2203620153,205,2203620153,22,03,62,01,53
558,008000,2203620153,242,2203620153,22,03,62,01,53
560,023000,2203620153,242,2203620153,22,03,62,01,53
562,034000,2203620153,242,2203620153,22,03,62,01,53
564,036000,2203620153,242,2203620153,22,03,62,01,53


In [7]:
# Identify single-unit long- and short-haul trucks (52, 53) , with ONI Process (92)
# This should inherently occur on roadtypes = 01, but this is also specified 
df.loc[(df.SCC_VehicleCode.isin(['52','53'])) & (df['SCC_RoadType'] == '01') & (df['SCC_ProcessCode'] == '92'), 'Surrogate'] = 903

In [8]:
# Identify combination long- and short-haul trucks (61,62) , with ONI Process (92)
# This should inherently occur on roadtypes = 01, but this is also specified 
df.loc[(df.SCC_VehicleCode.isin(['61','62'])) & (df['SCC_RoadType'] == '01') & (df['SCC_ProcessCode'] == '92'), 'Surrogate'] = 904

In [9]:
# replace extended idling surrogate 205 with modified 
df.loc[df['Surrogate'].eq('205'), 'Surrogate'] = '905'

In [10]:
# Replace surrogate 242 or 244 -> 238 for vehicle types 61 & 62, all roads except '01', processes 40 & 72
# 40 being brakewear & tirwear, 72 being exhaust  
df.loc[
    (df.SCC_VehicleCode.isin(['61','62'])) &
    (df['SCC_RoadType'] != '01') &
    (df['SCC_ProcessCode'].isin(['40','72'])) &
    (df['Surrogate'].isin({'242','244'})),
    'Surrogate'
] = 238


In [11]:
df[(df.SCC_VehicleCode.isin(['52','53','61','62'])) & (df['SCC_RoadType'] == '01')]

,FIPS,SCC,Surrogate,SCC1,SCC_mobileSource,SCC_fuelType,SCC_VehicleCode,SCC_RoadType,SCC_ProcessCode
0,000000,2202620153,905,2202620153,22,02,62,01,53
1,000000,2202620191,905,2202620191,22,02,62,01,91
11,000000,2201520162,239,2201520162,22,01,52,01,62
12,000000,2201530162,239,2201530162,22,01,53,01,62
14,000000,2201610162,239,2201610162,22,01,61,01,62
22,000000,2202520162,239,2202520162,22,02,52,01,62
24,000000,2202610162,239,2202610162,22,02,61,01,62
28,000000,2202620162,242,2202620162,22,02,62,01,62
34,000000,2201520172,306,2201520172,22,01,52,01,72
35,000000,2201530172,306,2201530172,22,01,53,01,72


In [12]:
# QAQC
df[(df.SCC_VehicleCode.isin(['52','53'])) & (df['SCC_RoadType'] == '01') & (df['SCC_ProcessCode'] == '92')]

,FIPS,SCC,Surrogate,SCC1,SCC_mobileSource,SCC_fuelType,SCC_VehicleCode,SCC_RoadType,SCC_ProcessCode
387,000000,2201520192,903,2201520192,22,01,52,01,92
388,000000,2201530192,903,2201530192,22,01,53,01,92
394,000000,2202520192,903,2202520192,22,02,52,01,92
395,000000,2202530192,903,2202530192,22,02,53,01,92
484,000000,2203520192,903,2203520192,22,03,52,01,92
499,000000,2203530192,903,2203530192,22,03,53,01,92


In [13]:
# QAQC
df[(df.SCC_VehicleCode.isin(['61','62'])) & (df['SCC_RoadType'] == '01') & (df['SCC_ProcessCode'] == '92')]

,FIPS,SCC,Surrogate,SCC1,SCC_mobileSource,SCC_fuelType,SCC_VehicleCode,SCC_RoadType,SCC_ProcessCode
390,000000,2201610192,904,2201610192,22,01,61,01,92
397,000000,2202610192,904,2202610192,22,02,61,01,92
398,000000,2202620192,904,2202620192,22,02,62,01,92
529,000000,2203610192,904,2203610192,22,03,61,01,92
566,000000,2203620192,904,2203620192,22,03,62,01,92


In [14]:
# QAQC
df[(df.SCC_VehicleCode.isin(['62'])) & (df['SCC_RoadType'] == '01') & (df['SCC_ProcessCode'] == '53')]

,FIPS,SCC,Surrogate,SCC1,SCC_mobileSource,SCC_fuelType,SCC_VehicleCode,SCC_RoadType,SCC_ProcessCode
0,000000,2202620153,905,2202620153,22,02,62,01,53
371,008000,2202620153,242,2202620153,22,02,62,01,53
373,023000,2202620153,242,2202620153,22,02,62,01,53
375,034000,2202620153,242,2202620153,22,02,62,01,53
377,036000,2202620153,242,2202620153,22,02,62,01,53
542,000000,2203620153,905,2203620153,22,03,62,01,53
558,008000,2203620153,242,2203620153,22,03,62,01,53
560,023000,2203620153,242,2203620153,22,03,62,01,53
562,034000,2203620153,242,2203620153,22,03,62,01,53
564,036000,2203620153,242,2203620153,22,03,62,01,53


In [15]:
# remove added columns 
df= df[['FIPS','SCC','Surrogate']]

In [18]:
# save as txt file. 
df.to_csv(r'NewActivityInputs/mgref_onroad_MOVES3_18nov2022_v4_LOCUSIdling_HDVSpatial.txt', index=False,header=None, sep=',', )

In [17]:
df

,FIPS,SCC,Surrogate
0,000000,2202620153,905
1,000000,2202620191,905
2,000000,2202420172,259
3,000000,2203420172,259
4,000000,2201110162,239
...,...,...,...
598,000000,2209520540,244
599,000000,2209530540,244
600,000000,2209540540,244
601,000000,2209610540,238
